# 06 — Results registry and manuscript table inventory

Indexes every saved visualization/table with a hash so manuscript figures can be traced to an exact output.

Every displayed denominator and paper-facing visual is also saved under `outputs/visualization/`. Empty or under-supported analyses remain visible as audit rows; they are never silently removed.

In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "paper1_qc").exists():
            return candidate
    raise FileNotFoundError("Open Jupyter from inside paper1_pipeline_rebuilt.")

ROOT = find_project_root()
CONFIG = ROOT / "config" / "project.yaml"
OUTPUT = ROOT / "outputs"
VIZ_ROOT = OUTPUT / "visualization"
sys.path.insert(0, str(ROOT / "src"))

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def read_stage(relative_without_suffix):
    stem = OUTPUT / relative_without_suffix
    parquet = stem.with_suffix(".parquet")
    csv = stem.with_suffix(".csv")
    if parquet.exists():
        return pd.read_parquet(parquet)
    if csv.exists():
        try:
            return pd.read_csv(csv)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()
    raise FileNotFoundError(f"Missing required stage table: {parquet} or {csv}")

def run_cli(*arguments):
    command = [sys.executable, "-m", "paper1_qc.cli", "--config", str(CONFIG), *arguments]
    print("RUN:", " ".join(map(str, command)))
    subprocess.run(command, cwd=ROOT, check=True)

def save_table(frame, folder, name):
    target = VIZ_ROOT / folder
    target.mkdir(parents=True, exist_ok=True)
    path = target / f"{name}.csv"
    frame.to_csv(path, index=False)
    print("TABLE:", path.relative_to(ROOT), f"({len(frame):,} rows)")
    return path

def save_figure(fig, folder, name):
    target = VIZ_ROOT / folder
    target.mkdir(parents=True, exist_ok=True)
    png = target / f"{name}.png"
    svg = target / f"{name}.svg"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print("FIGURE:", png.relative_to(ROOT))
    return png, svg

assert CONFIG.exists(), "Copy config/project.example.yaml to config/project.yaml and review it."
print("Project:", ROOT)
print("Config:", CONFIG)
print("Visualization outputs:", VIZ_ROOT)


In [ ]:
files = sorted(path for path in VIZ_ROOT.rglob("*") if path.is_file())
rows = []
for path in files:
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    rows.append({
        "relative_path": path.relative_to(ROOT).as_posix(),
        "extension": path.suffix.lower(),
        "size_bytes": path.stat().st_size,
        "sha256": digest,
    })
registry = pd.DataFrame(rows)
save_table(registry, "06_registry", "visualization_output_registry")
display(registry)


In [ ]:
figure_candidates = pd.DataFrame([
    {"manuscript_role": "Segmentation supplement", "source": "outputs/visualization/01_segmentation/speech_fraction_ecdf.png", "status": "candidate"},
    {"manuscript_role": "Goal 1 support panel", "source": "outputs/visualization/02_goal1/metric_support_fraction.png", "status": "candidate"},
    {"manuscript_role": "Goal 1 acquisition distributions", "source": "outputs/visualization/02_goal1/raw_distributions__<family>.png", "status": "family facets"},
    {"manuscript_role": "Goal 2 persistence", "source": "outputs/visualization/03_goal2/participant_rank_persistence.png", "status": "candidate"},
    {"manuscript_role": "Goal 3 structure", "source": "outputs/visualization/04_goal3/correlation_and_support_matrices.png", "status": "candidate"},
    {"manuscript_role": "Goal 3 Rest sensitivity", "source": "outputs/visualization/04_goal3/rest_reference_level_comparison.png", "status": "supplement candidate"},
    {"manuscript_role": "Goal 4 family validity", "source": "outputs/visualization/05_goal4/four_ra_alignment_and_denominators.png", "status": "candidate"},
    {"manuscript_role": "Goal 4 label-system comparison", "source": "outputs/visualization/05_goal4/four_ra_minus_two_ra_delta_auc.png", "status": "candidate if estimable"},
])
save_table(figure_candidates, "06_registry", "manuscript_figure_candidates")
display(figure_candidates)
